# Fine-tuning del NLU de la plataforma de taxis (Qwen2.5 + LoRA → Ollama)

Este cuaderno entrena el modelo que usa el chatbot y lo deja listo para Ollama.
Con una **GPU T4 gratuita** tarda unos 15–20 minutos en total.

1. `Entorno de ejecución → Cambiar tipo de entorno → T4 GPU`.
2. Ejecuta las celdas en orden. En la segunda te pedirá el zip del proyecto.
3. Al final se descarga `pids-nlu-modelo.zip`: descomprímelo en la carpeta `llm/modelo/` del proyecto y haz `docker compose up`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Subir el proyecto

In [ ]:
from google.colab import files

subido = files.upload()          # elige el zip del proyecto (PIDS-plataforma-taxis-*.zip)
nombre = next(iter(subido))
!unzip -q -o "$nombre" -d /content/proyecto
%cd /content/proyecto/pids

In [ ]:
!pip install -q -r llm/requirements.txt

## 2. Dataset
Frases sintéticas en español con valores reales de los datos. El conjunto de **test** usa plantillas que el modelo no ve al entrenar.

In [ ]:
!python llm/generar_dataset.py

## 3. Entrenamiento LoRA
Por defecto `Qwen/Qwen2.5-1.5B-Instruct`, 2 épocas. Para un modelo más ligero (más rápido en CPU, algo menos preciso) usa
`--modelo-base Qwen/Qwen2.5-0.5B-Instruct --epocas 3`.

In [ ]:
!python llm/entrenar.py

## 4. Exportar a GGUF + Modelfile

In [ ]:
!python llm/exportar.py --tipo q8_0

## 5. Evaluación con Ollama (reglas vs modelo base vs modelo afinado)
Instala Ollama en la máquina de Colab, crea el modelo afinado y lo compara con el base y con las reglas sobre las frases de test.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
import subprocess
import time

servidor = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
!cd llm/modelo && ollama create pids-nlu -f Modelfile
!ollama pull qwen2.5:1.5b

In [ ]:
!python llm/evaluar.py --motor reglas \
    --motor ollama --modelo qwen2.5:1.5b --base qwen2.5:1.5b \
    --motor ollama --modelo pids-nlu \
    --json llm/modelo/informe_evaluacion.json

In [ ]:
!ollama run pids-nlu "el viaje 411 tenía mal la tarifa, eran 23,50 porque el taxímetro falló"

## 6. Descargar el modelo

In [ ]:
!cd llm/modelo && zip -q -r /content/pids-nlu-modelo.zip .
files.download("/content/pids-nlu-modelo.zip")